## Código inicial

In [ ]:
import os

from_drive = True  # same flag you use everywhere

if os.environ.get("ATLAS_BOOTSTRAPPED") != "1":
    # ---------- GIT ON COLAB ONLY ----------
    try:
        from google.colab import userdata

        git_token = userdata.get('gitToken')
        git_user = userdata.get('gitUser')
        git_url = f'https://{git_token}@github.com/rene-aum/Atlas.git'
        branch_to_pull = 'dev'

        os.chdir('/content')

        if not os.path.isdir('Atlas'):
            !git clone {git_url}

        %cd Atlas
        !git fetch origin {branch_to_pull}
        !git checkout {branch_to_pull}
        !git pull origin {branch_to_pull}

        !pip install -r PipelinesConsumo/src/requirements.txt
        %cd PipelinesConsumo

    except Exception as e:
        print(e)
        print('Running in other environment not colab probably!')

    # ---------- DRIVE + SHEETS ----------
    if from_drive:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        import gspread
        from google.auth import default
        from gspread_dataframe import set_with_dataframe
        import gdown

        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)

        creds, _ = default()
        gc = gspread.authorize(creds)

    os.environ["ATLAS_BOOTSTRAPPED"] = "1"
else:
    print("Bootstrap already done, assuming orchestrator ran it.")

In [ ]:
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
import sys
sys.path.append('..')
sys.path.append('../..')
from utils.utils import (get_dates_dataframe,
                       add_year_week,
                       custom_read,
                       process_columns)
from PipelinesConsumo.src.rawAtlas import RawAtlas
from PipelinesConsumo.src.processedAtlas import ProcessedAtlas
from src.transformed import Transformed
from utils.drive_toolbox import(from_drive_to_local,
                             get_last_modification_date_drive,
                             create_sheets_in_drive_folder,
                             update_sheets_in_drive_folder,
                             read_from_google_sheets,
                             list_file_ids_for_drive_folder,
                             create_csv_file_in_drive_folder,
                             write_csv_to_drive,
                             read_csv_from_drive,
                             send_google_chat_notification,
                             update_sheets_in_drive_folder_chunked)
from src.constants import (atlas_raw_output_folder_id,
                           atlas_consumo_output_folder_id,
                           consumo_sheets_ids_dict,
                           data_source_folder_id,
                           raw_output_ids,
                           folder_id_bauto_gabo,
                           id_reporte_ventas,
                           id_edas_referenciados
                           )

import time
warnings.filterwarnings('ignore')

# Configuración de Proyectos

In [ ]:
proyectos = {
    1: {
        'nombre': 'AcClientes',
        'id_prod': r'1Re7omesAjvf8hoQ9n_BTCMBUnM37YOj6VOJTZHs3I6I',
        'id_dev':  r'1ppBSiQ52nSMiHNBMRwk70BFONHp_I6yZPHrFxVyEo_Q',
        'hoja': 'Hoja 1',
        'col_llave': 'id_am',
        'hoja_escritura': 'AcClientes-Data',
        'hoja_resumen':   'AcClientes'
    },
    2: {
        'nombre': 'AcPedidos',
        'id_prod': r'1aBfKoX6d9BEPhhYg-DDuvC5QFcd3EQtXb0YrU7n4nYM',
        'id_dev':  r'1yF9XA0PSdZLExQStGcHFSpUI56dJYE50aaTlFEV8uCk',
        'hoja': 'Hoja 1',
        'col_llave': 'sf_order_id',
        'hoja_escritura': 'AcPedidos-Data',
        'hoja_resumen':   'AcPedidos'
    },
    3: {
        'nombre': 'AcVisitasUnicas',
        'id_prod': r'1jmbOj9a7-dnXUgH276lJ-blP-auDdWGemzia1RTsNEo',
        'id_dev':  r'1ELHlm0yWNsRn7kmKXpTQBhvSlvQeBYdTglPjG5DxKTQ',
        'hoja': 'Hoja 1',
        'col_llave': 'date',
        'hoja_escritura': 'AcVisitasUnicas-Data',
        'hoja_resumen':   'AcVisitasUnicas'
    },
    4: {
        'nombre': 'AcPublicacionesCanceladas',
        'id_prod': r'1AZdaqfSw6QV9eXNRYgnuizvSsWRcyBUJ4IgsZkcw9gk',
        'id_dev':  r'1GUgdzRhmmec2L_HqJBVhheK2YUMreTQ1E5npTaE2fuY',
        'hoja': 'Hoja 1',
        'col_llave': 'sku',
        'hoja_escritura': 'AcPublicacionesCanceladas-Data',
        'hoja_resumen':   'AcPublicacionesCanceladas'
    },
    5: {
        'nombre': 'AcPublicaciones',
        'id_prod': r'1NIvj4zjUO9N4fsiW1I85VI2RfTCT54nvbBON_l0xeYI',
        'id_dev':  r'18HOA3JS33OpICwY_7XgJpXh49bcn3qFLEPx723c1bd4',
        'hoja': 'Hoja 1',
        'col_llave': 'sku',
        'hoja_escritura': 'AcPublicaciones-Data',
        'hoja_resumen':   'AcPublicaciones'
    },
    6: {
        'nombre': 'AcAdobeFunnelCompradorTotal',
        'id_prod': r'1XEThwBlrkC01n3ocAtLXuHdattooWKmK-yZEiYzkr-8',
        'id_dev':  r'1pLJVFQoHO5uUVXlPwkPiIKP1ugJjva-kvypMV_U4HBI',
        'hoja': 'Hoja 1',
        'col_llave': 'date',
        'hoja_escritura': 'AcAdobeFunnelCompradorTotal-Data',
        'hoja_resumen':   'AcAdobeFunnelCompradorTotal'
    },
    7: {
        'nombre': 'AcAdobeFunnelCompradorUsuario',
        'id_prod': r'1iE0CLKpfje1BV42EH6vnj58nUes3T1RpVvB_4kXCawM',
        'id_dev':  r'1kwtOw5EhTnsjODtS7xnVqmp5R7O2VHYzV6dJu8hv59M',
        'hoja': 'Hoja 1',
        'col_llave': 'date',
        'hoja_escritura': 'AcAdobeFunnelCompradorUsuario-Data',
        'hoja_resumen':   'AcAdobeFunnelCompradorUsuario'
    },
    8: {
        'nombre': 'AcAdobeFunnelVendedorTotal',
        'id_prod': r'1prXQStwzaAMiPwSz24Fi2sfetuO24l6YyYA1Z_7xVpE',
        'id_dev':  r'1lfQdlkl_pYLrD6GFJrOWFJaFaLEZvQygAhAnkljxspg',
        'hoja': 'Hoja 1',
        'col_llave': 'date',
        'hoja_escritura': 'AcAdobeFunnelVendedorTotal-Data',
        'hoja_resumen':   'AcAdobeFunnelVendedorTotal'
    },
    9: {
        'nombre': 'AcAdobeFunnelVendedorUsuario',
        'id_prod': r'15S67jk2ZwRjhpaicksWPMwk6HPV2sk4rtSEkLHjsTr8',
        'id_dev':  r'18NezvGLza7Xs1x0mLTzDf__qW3w0BS3u_IrFqtDeu0Q',
        'hoja': 'Hoja 1',
        'col_llave': 'date',
        'hoja_escritura': 'AcAdobeFunnelVendedorUsuario-Data',
        'hoja_resumen':   'AcAdobeFunnelVendedorUsuario'
    }
}

print(f"Proyectos configurados: {len(proyectos)}")

In [ ]:
import re
import pandas as pd
import numpy as np
from collections import Counter


# ── LIMPIEZA DE CARACTERES Y TIPOS ───────────────────────────────────────────

def limpiar_pipeline_completo(df):
    traducciones = {
        '√°': 'á', '√©': 'é', '√≠': 'í', '√≥': 'ó', '√∫': 'ú',
        '√±': 'ñ', '√º': 'ü', '√ﾁ': 'Á', '√ﾉ': 'É', '√ﾍ': 'Í',
        '√ﾓ': 'Ó', '√ﾚ': 'Ú', '√ﾑ': 'Ñ', '√Ë': 'Ñ', '√Â': 'É',
        '√Ç': 'Í', '√ç': 'í', '√Ì': 'Ó', '√Å': 'Á', '√Ö': 'Ú',
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã±': 'ñ', 'Ã\"': 'Ó', 'Ã‰': 'É', 'Ã ': 'À', 'Ã'': 'Ñ',
        'Ã\x9a': 'Ú', 'Ã\x93': 'Ó', 'Ã\x81': 'Á', 'Ã\x89': 'É', 'Ã\x8d': 'Í',
        'Ã"': 'Ó', 'ÃŠ': 'Ú',
        'Å': 'Ú',
        'â€"': '—', 'â€¦': '...', '\xa0': ' ',
        'Â': '',
    }
    patron = re.compile(
        '|'.join(re.escape(k) for k in sorted(traducciones, key=len, reverse=True))
    )
    PROTEGIDOS = set('√ÃÂÅ')

    def fix_completo(val):
        if not isinstance(val, str):
            return val
        val = patron.sub(lambda m: traducciones[m.group(0)], val)
        if any(c in val for c in ('√', 'Ã', 'â€')) and not any(c in val for c in PROTEGIDOS):
            for encoding in ('macroman', 'latin-1'):
                try:
                    candidato = val.encode(encoding).decode('utf-8')
                    if not re.search(r'[√Ã]|â€', candidato):
                        val = candidato
                        break
                except (UnicodeEncodeError, UnicodeDecodeError):
                    continue
        return val

    for col in df.columns:
        if df[col].dtype == 'object':
            mask = df[col].notna()
            df.loc[mask, col] = df[col].loc[mask].apply(fix_completo).str.strip()
        try:
            temp_num = pd.to_numeric(df[col], errors='coerce')
            if not temp_num.isna().all():
                df[col] = temp_num.fillna(0).round(0).astype(int)
        except:
            continue
    return df


def normalizar_pais(df):
    target_col = next((c for c in df.columns if c.lower() == 'country'), None)
    if target_col:
        df[target_col] = df[target_col].str.upper().str.strip()
        df[target_col] = df[target_col].replace(['MEXICO', 'MÉXICO', 'MÉXICO', 'Mexico'], 'MX')
        df[target_col] = df[target_col].fillna('').replace(['NAN', 'NONE', 'N/A'], '')
    return df


def normalizar_telefono(df):
    target_col = next((c for c in df.columns if c.lower() == 'phone'), None)
    if target_col:
        df[target_col] = df[target_col].fillna('').astype(str)
        df[target_col] = df[target_col].str.replace(r'\D', '', regex=True)
        df[target_col] = df[target_col].str[-10:]
        df[target_col] = df[target_col].replace(['nan', 'None', 'NAN'], '')
    return df


def normalizar_a_mayusculas(df):
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.upper().str.strip()
    return df


# ── HELPERS DE NORMALIZACIÓN ─────────────────────────────────────────────────

def limpiar_llave_int(serie):
    def procesar_valor(x):
        if pd.isna(x) or str(x).strip().lower() in ['nan', 'none', '']:
            return "0"
        val_str = str(x).strip()
        try:
            f = float(val_str)
            return str(int(f)) if f.is_integer() else str(f)
        except ValueError:
            return val_str
    return serie.apply(procesar_valor)


def forzar_int_str(valor):
    if pd.isna(valor) or str(valor).lower() in ['nan', 'none', '']:
        return ""
    try:
        f_val = float(valor)
        return str(int(f_val)) if f_val == int(f_val) else str(f_val)
    except:
        return str(valor).strip()


print("✅ Funciones de limpieza y normalización cargadas.")

# Validación Automática — Todos los Proyectos

In [ ]:
import traceback
import time

ID_HOJA_RESULTADOS = '15SPKEg41SyiDNWb06WkTi0cJj8Sn9Vcx9F1uszdV2Ik'

def _ts():
    return time.strftime('%H:%M:%S')

resumen_global = []

for opcion, config in proyectos.items():
    nombre         = config['nombre']
    col_llave      = config['col_llave']
    hoja_escritura = config['hoja_escritura']
    hoja_resumen   = config['hoja_resumen']
    t0             = time.time()

    print(f"\n{'='*70}")
    print(f"  [{opcion}/{len(proyectos)}] {nombre}  —  {_ts()}")
    print('='*70)

    resultado = {
        'proyecto':                  nombre,
        'identicos':                 False,
        'filas_prod':                0,
        'filas_dev':                 0,
        'dups_prod':                 0,
        'dups_dev':                  0,
        'solo_en_prod':              0,
        'solo_en_dev':               0,
        'registros_con_diferencias': 0,
        'tipos_distintos':           0,
        'cols_solo_prod':            [],
        'cols_solo_dev':             [],
        'error':                     None,
    }

    try:
        # ── CARGA ─────────────────────────────────────────────────────────
        print(f"  [{_ts()}] Cargando PROD...")
        prod = read_from_google_sheets(gc, config['id_prod'], config['hoja'])
        print(f"  [{_ts()}] Cargando DEV...")
        dev  = read_from_google_sheets(gc, config['id_dev'],  config['hoja'])
        prod.columns = [str(c) for c in prod.columns]
        dev.columns  = [str(c) for c in dev.columns]
        print(f"  [{_ts()}] Datos cargados — PROD: {len(prod)} filas  DEV: {len(dev)} filas")

        # ── LIMPIEZA ──────────────────────────────────────────────────────
        print(f"  [{_ts()}] Limpiando y normalizando...")
        prod = limpiar_pipeline_completo(prod)
        dev  = limpiar_pipeline_completo(dev)
        prod = normalizar_pais(prod)
        dev  = normalizar_pais(dev)
        prod = normalizar_telefono(prod)
        dev  = normalizar_telefono(dev)
        prod = normalizar_a_mayusculas(prod)
        dev  = normalizar_a_mayusculas(dev)

        # ── 0. IGUALDAD DIRECTA ───────────────────────────────────────────
        resultado['identicos'] = prod.equals(dev)
        print(f"  ¿DataFrames idénticos?: {resultado['identicos']}")

        # ── 1. COLUMNAS ───────────────────────────────────────────────────
        cols_prod = set(prod.columns)
        cols_dev  = set(dev.columns)
        resultado['cols_solo_prod'] = sorted(cols_prod - cols_dev)
        resultado['cols_solo_dev']  = sorted(cols_dev  - cols_prod)
        if resultado['cols_solo_prod']:
            print(f"  Columnas solo en PROD: {resultado['cols_solo_prod']}")
        if resultado['cols_solo_dev']:
            print(f"  Columnas solo en DEV:  {resultado['cols_solo_dev']}")

        cmp_tipos = pd.concat([prod.dtypes, dev.dtypes], axis=1, keys=['prod', 'dev'])
        resultado['tipos_distintos'] = int((cmp_tipos['prod'] != cmp_tipos['dev']).sum())
        if resultado['tipos_distintos']:
            print(f"  Columnas con tipo distinto: {resultado['tipos_distintos']}")

        # ── 2. FILAS ──────────────────────────────────────────────────────
        resultado['filas_prod'] = len(prod)
        resultado['filas_dev']  = len(dev)
        print(f"  Filas — PROD: {resultado['filas_prod']}  DEV: {resultado['filas_dev']}")

        # ── 3 & 4. DUPLICADOS + CONTENIDO ────────────────────────────────
        registros_errores = []

        if opcion <= 5:
            ids_prod_clean = limpiar_llave_int(prod[col_llave])
            ids_dev_clean  = limpiar_llave_int(dev[col_llave])

            resultado['dups_prod'] = int(ids_prod_clean.duplicated().sum())
            resultado['dups_dev']  = int(ids_dev_clean.duplicated().sum())
            print(f"  Duplicados — PROD: {resultado['dups_prod']}  DEV: {resultado['dups_dev']}")

            if opcion == 4:
                ids_prod    = Counter(ids_prod_clean)
                ids_dev     = Counter(ids_dev_clean)
                solo_prod   = list((ids_prod - ids_dev).elements())
                solo_dev    = list((ids_dev  - ids_prod).elements())
                ids_comunes = set(ids_prod.keys()) & set(ids_dev.keys())
            else:
                ids_prod    = set(ids_prod_clean)
                ids_dev     = set(ids_dev_clean)
                solo_prod   = list(ids_prod - ids_dev)
                solo_dev    = list(ids_dev  - ids_prod)
                ids_comunes = ids_prod & ids_dev

            resultado['solo_en_prod'] = len(solo_prod)
            resultado['solo_en_dev']  = len(solo_dev)

            for id_val in solo_prod:
                registros_errores.append({col_llave: id_val, 'Error': 'SOLO_EN_PROD', 'Flag': 1})
            for id_val in solo_dev:
                registros_errores.append({col_llave: id_val, 'Error': 'SOLO_EN_DEV',  'Flag': 1})

            print(f"  [{_ts()}] Comparando contenido ({len(ids_comunes)} registros comunes)...")
            prod_comun = prod[limpiar_llave_int(prod[col_llave]).isin(ids_comunes)].copy()
            dev_comun  = dev[limpiar_llave_int(dev[col_llave]).isin(ids_comunes)].copy()
            prod_comun[col_llave] = limpiar_llave_int(prod_comun[col_llave])
            dev_comun[col_llave]  = limpiar_llave_int(dev_comun[col_llave])
            columnas_a_comparar   = [c for c in prod.columns if c != col_llave and c in dev.columns]

            print(f"\n  {'COLUMNA':<30} | {'ESTADO':<20} | DETALLE")
            print(f"  {'-'*100}")

            if opcion == 4:
                comparativo = pd.merge(prod_comun, dev_comun, on=col_llave,
                                       suffixes=('_prod', '_dev'), how='inner')
                for col in columnas_a_comparar:
                    cp, cd = f"{col}_prod", f"{col}_dev"
                    v_p           = comparativo[cp].apply(forzar_int_str)
                    v_d           = comparativo[cd].apply(forzar_int_str)
                    ids_con_match = set(comparativo.loc[v_p == v_d, col_llave])

                    checks = [
                        (comparativo[cp].notna() & (comparativo[cp] != '') &
                         (comparativo[cd].isna()  | (comparativo[cd] == '')), 'FALTA_EN_DEV',  'Falta en DEV'),
                        (comparativo[cd].notna() & (comparativo[cd] != '') &
                         (comparativo[cp].isna()  | (comparativo[cp] == '')), 'FALTA_EN_PROD', 'Falta en PROD'),
                        ((comparativo[cp].notna() & (comparativo[cp] != '')) &
                         (comparativo[cd].notna() & (comparativo[cd] != '')) &
                         (v_p != v_d), 'DIFERENTE', 'Valores distintos'),
                    ]
                    for mascara, sufijo, msg in checks:
                        ids_fallo = set(comparativo.loc[mascara, col_llave]) - ids_con_match
                        if ids_fallo:
                            for id_val in ids_fallo:
                                registros_errores.append({col_llave: id_val,
                                                          'Error': f"{col}_{sufijo}", 'Flag': 1})
                            ej_id  = next(iter(ids_fallo))
                            ej_idx = comparativo[comparativo[col_llave] == ej_id].index[0]
                            print(f"  {col:<30} | {msg:<20} | {len(ids_fallo)} llaves "
                                  f"(Ej {ej_id}: {forzar_int_str(comparativo.loc[ej_idx, cp])} "
                                  f"vs {forzar_int_str(comparativo.loc[ej_idx, cd])})")
            else:
                comparativo = pd.merge(prod_comun, dev_comun, on=col_llave, suffixes=('_prod', '_dev'))
                for col in columnas_a_comparar:
                    cp, cd = f"{col}_prod", f"{col}_dev"
                    v_p    = comparativo[cp].apply(forzar_int_str)
                    v_d    = comparativo[cd].apply(forzar_int_str)

                    checks = [
                        (comparativo[cp].notna() & (comparativo[cp] != '') &
                         (comparativo[cd].isna()  | (comparativo[cd] == '')), 'FALTA_EN_DEV',  'Falta en DEV'),
                        (comparativo[cd].notna() & (comparativo[cd] != '') &
                         (comparativo[cp].isna()  | (comparativo[cp] == '')), 'FALTA_EN_PROD', 'Falta en PROD'),
                        ((comparativo[cp].notna() & (comparativo[cp] != '')) &
                         (comparativo[cd].notna() & (comparativo[cd] != '')) &
                         (v_p != v_d), 'DIFERENTE', 'Valores distintos'),
                    ]
                    for mascara, sufijo, msg in checks:
                        if mascara.any():
                            ids_fallo = comparativo.loc[mascara, col_llave].tolist()
                            for id_val in ids_fallo:
                                registros_errores.append({col_llave: id_val,
                                                          'Error': f"{col}_{sufijo}", 'Flag': 1})
                            idx = mascara.idxmax()
                            print(f"  {col:<30} | {msg:<20} | {len(ids_fallo)} filas "
                                  f"(Ej {comparativo.loc[idx, col_llave]}: "
                                  f"{forzar_int_str(comparativo.loc[idx, cp])} "
                                  f"vs {forzar_int_str(comparativo.loc[idx, cd])})")

        else:
            # ── LLAVE COMPUESTA (date + id_am) — opcion >= 6 ──────────────
            COL_DATE = 'date'
            COL_IDAM = 'id_am'

            prod[COL_DATE] = prod[COL_DATE].astype(str).str.strip()
            dev[COL_DATE]  = dev[COL_DATE].astype(str).str.strip()
            prod[COL_IDAM] = prod[COL_IDAM].apply(forzar_int_str)
            dev[COL_IDAM]  = dev[COL_IDAM].apply(forzar_int_str)

            prod_con = prod[prod[COL_IDAM] != ''].copy()
            prod_vac = prod[prod[COL_IDAM] == ''].copy()
            dev_con  = dev[dev[COL_IDAM]   != ''].copy()
            dev_vac  = dev[dev[COL_IDAM]   == ''].copy()

            def llave_str(df):
                return df[COL_DATE].astype(str) + '||' + df[COL_IDAM].astype(str)

            counter_prod_con = Counter(llave_str(prod_con))
            counter_dev_con  = Counter(llave_str(dev_con))
            counter_prod_vac = Counter(prod_vac[COL_DATE])
            counter_dev_vac  = Counter(dev_vac[COL_DATE])

            llaves_prod_todas = llave_str(prod_con).tolist() + [f"{d}||" for d in prod_vac[COL_DATE]]
            llaves_dev_todas  = llave_str(dev_con).tolist()  + [f"{d}||" for d in dev_vac[COL_DATE]]

            resultado['dups_prod'] = int(pd.Series(llaves_prod_todas).duplicated().sum())
            resultado['dups_dev']  = int(pd.Series(llaves_dev_todas).duplicated().sum())
            print(f"  Duplicados — PROD: {resultado['dups_prod']}  DEV: {resultado['dups_dev']}")

            solo_prod_con = list((counter_prod_con - counter_dev_con).elements())
            solo_dev_con  = list((counter_dev_con  - counter_prod_con).elements())
            solo_prod_vac = list((counter_prod_vac - counter_dev_vac).elements())
            solo_dev_vac  = list((counter_dev_vac  - counter_prod_vac).elements())

            resultado['solo_en_prod'] = len(solo_prod_con) + len(solo_prod_vac)
            resultado['solo_en_dev']  = len(solo_dev_con)  + len(solo_dev_vac)

            for llave in solo_prod_con:
                registros_errores.append({'llave': llave, 'Error': 'SOLO_EN_PROD', 'Flag': 1})
            for llave in solo_dev_con:
                registros_errores.append({'llave': llave, 'Error': 'SOLO_EN_DEV',  'Flag': 1})
            for dia in solo_prod_vac:
                registros_errores.append({'llave': f"{dia}||VACIO_EXTRA", 'Error': 'SOLO_EN_PROD', 'Flag': 1})
            for dia in solo_dev_vac:
                registros_errores.append({'llave': f"{dia}||VACIO_EXTRA", 'Error': 'SOLO_EN_DEV',  'Flag': 1})

            prod_con = prod_con.copy()
            dev_con  = dev_con.copy()
            prod_con['_llave'] = llave_str(prod_con)
            dev_con['_llave']  = llave_str(dev_con)

            llaves_comunes_con = set(counter_prod_con.keys()) & set(counter_dev_con.keys())
            dias_comunes_vac   = set(counter_prod_vac.keys()) & set(counter_dev_vac.keys())
            todas_las_fechas   = sorted(
                set(prod_con[COL_DATE]) | set(dev_con[COL_DATE]) |
                set(prod_vac[COL_DATE]) | set(dev_vac[COL_DATE])
            )
            columnas_a_comparar = [
                c for c in prod.columns
                if c not in (COL_DATE, COL_IDAM, '_llave') and c in dev.columns
            ]

            total_fechas  = len(todas_las_fechas)
            paso_progreso = max(1, total_fechas // 10)  # log cada ~10%
            print(f"  [{_ts()}] Fechas a procesar: {total_fechas}")
            print(f"\n  {'COLUMNA':<30} | {'ESTADO':<20} | DETALLE")
            print(f"  {'-'*100}")

            for i, dia in enumerate(todas_las_fechas, 1):
                if i == 1 or i % paso_progreso == 0 or i == total_fechas:
                    pct = int(i / total_fechas * 100)
                    print(f"  [{_ts()}] Fecha {i}/{total_fechas} ({pct}%) — {dia}")

                p_dia = prod_con[(prod_con[COL_DATE] == dia) & (prod_con['_llave'].isin(llaves_comunes_con))]
                d_dia = dev_con[(dev_con[COL_DATE]   == dia) & (dev_con['_llave'].isin(llaves_comunes_con))]

                if not p_dia.empty and not d_dia.empty:
                    comp = pd.merge(p_dia, d_dia, on='_llave', suffixes=('_prod', '_dev'), how='inner')
                    for col in columnas_a_comparar:
                        cp, cd = f"{col}_prod", f"{col}_dev"
                        v_p           = comp[cp].apply(forzar_int_str)
                        v_d           = comp[cd].apply(forzar_int_str)
                        ids_con_match = set(comp.loc[v_p == v_d, '_llave'])

                        checks = [
                            (comp[cp].notna() & (comp[cp] != '') &
                             (comp[cd].isna()  | (comp[cd] == '')), 'FALTA_EN_DEV',  'Falta en DEV'),
                            (comp[cd].notna() & (comp[cd] != '') &
                             (comp[cp].isna()  | (comp[cp] == '')), 'FALTA_EN_PROD', 'Falta en PROD'),
                            ((comp[cp].notna() & (comp[cp] != '')) &
                             (comp[cd].notna() & (comp[cd] != '')) &
                             (v_p != v_d), 'DIFERENTE', 'Valores distintos'),
                        ]
                        for mascara, sufijo, msg in checks:
                            ids_fallo = set(comp.loc[mascara, '_llave']) - ids_con_match
                            if ids_fallo:
                                for id_val in ids_fallo:
                                    registros_errores.append({'llave': id_val,
                                                              'Error': f"{col}_{sufijo}", 'Flag': 1})
                                ej_id  = next(iter(ids_fallo))
                                ej_idx = comp[comp['_llave'] == ej_id].index[0]
                                print(f"  {col:<30} | {msg:<20} | {len(ids_fallo)} llaves "
                                      f"(Ej {ej_id}: {forzar_int_str(comp.loc[ej_idx, cp])} "
                                      f"vs {forzar_int_str(comp.loc[ej_idx, cd])})")

                if dia in dias_comunes_vac:
                    filas_p = prod_vac[prod_vac[COL_DATE] == dia]
                    filas_d = dev_vac[dev_vac[COL_DATE]   == dia]
                    for col in columnas_a_comparar:
                        vals_p    = filas_p[col].apply(forzar_int_str)
                        vals_d    = filas_d[col].apply(forzar_int_str)
                        set_p     = set(vals_p)
                        set_d     = set(vals_d)
                        hay_match = bool((set_p & set_d) - {''})
                        llave_rep = f"{dia}||VACIO"

                        if (vals_p != '').any() and (vals_d == '').all() and not hay_match:
                            registros_errores.append({'llave': llave_rep,
                                                      'Error': f"{col}_FALTA_EN_DEV", 'Flag': 1})
                            print(f"  {col:<30} | {'Falta en DEV':<20} | día {dia}")
                        elif (vals_d != '').any() and (vals_p == '').all() and not hay_match:
                            registros_errores.append({'llave': llave_rep,
                                                      'Error': f"{col}_FALTA_EN_PROD", 'Flag': 1})
                            print(f"  {col:<30} | {'Falta en PROD':<20} | día {dia}")
                        elif (set_p - set_d - {''}) and not hay_match:
                            registros_errores.append({'llave': llave_rep,
                                                      'Error': f"{col}_DIFERENTE", 'Flag': 1})
                            print(f"  {col:<30} | {'Valores distintos':<20} | día {dia}")

        # ── df_ids_diferencias ────────────────────────────────────────────
        idx_col = col_llave if opcion <= 5 else 'llave'

        if registros_errores:
            df_temp = pd.DataFrame(registros_errores)
            df_ids_diferencias = df_temp.pivot_table(
                index=idx_col, columns='Error', values='Flag', fill_value=0
            ).reset_index()
            cols_error = [c for c in df_ids_diferencias.columns if c != idx_col]
            df_ids_diferencias[cols_error] = df_ids_diferencias[cols_error].astype(int)
        else:
            df_ids_diferencias = pd.DataFrame(columns=[idx_col])

        # Enriquecimiento especial opcion 2
        if opcion == 2 and not df_ids_diferencias.empty:
            prod['sf_order_id'] = prod['sf_order_id'].astype(str)
            dev['sf_order_id']  = dev['sf_order_id'].astype(str)
            df_ids_diferencias['sf_order_id'] = df_ids_diferencias['sf_order_id'].astype(str)
            prod['fecha_de_creacion'] = pd.to_datetime(prod['fecha_de_creacion'], errors='coerce')
            dev['fecha_de_creacion']  = pd.to_datetime(dev['fecha_de_creacion'],  errors='coerce')
            merged_fecha = prod[['sf_order_id', 'fecha_de_creacion']].merge(
                dev[['sf_order_id', 'fecha_de_creacion']],
                on='sf_order_id', suffixes=('_prod', '_dev')
            )
            merged_fecha['fecha_creacion_dif_prod_menos_dev'] = (
                merged_fecha['fecha_de_creacion_prod'] - merged_fecha['fecha_de_creacion_dev']
            ).dt.days
            df_ids_diferencias = df_ids_diferencias.merge(
                merged_fecha[['sf_order_id', 'fecha_creacion_dif_prod_menos_dev']],
                on='sf_order_id', how='left'
            )

        resultado['registros_con_diferencias'] = len(df_ids_diferencias)
        print(f"\n  Registros con diferencias: {len(df_ids_diferencias)}")

        # ── ESCRITURA DETALLE EN SHEETS ───────────────────────────────────
        print(f"  [{_ts()}] Escribiendo detalle en Sheets...")
        update_sheets_in_drive_folder_chunked(gc, ID_HOJA_RESULTADOS, hoja_escritura, df_ids_diferencias)
        print(f"  [{_ts()}] ✅ Escrito detalle en '{hoja_escritura}'")

        # ── ESCRITURA RESUMEN EN SHEETS ───────────────────────────────────
        df_resumen = pd.DataFrame([
            {'Métrica': 'Proyecto',                  'Valor': resultado['proyecto']},
            {'Métrica': 'Idénticos',                 'Valor': 'SI' if resultado['identicos'] else 'NO'},
            {'Métrica': 'Filas PROD',                'Valor': resultado['filas_prod']},
            {'Métrica': 'Filas DEV',                 'Valor': resultado['filas_dev']},
            {'Métrica': 'Duplicados PROD',            'Valor': resultado['dups_prod']},
            {'Métrica': 'Duplicados DEV',             'Valor': resultado['dups_dev']},
            {'Métrica': 'Solo en PROD',              'Valor': resultado['solo_en_prod']},
            {'Métrica': 'Solo en DEV',               'Valor': resultado['solo_en_dev']},
            {'Métrica': 'Registros con diferencias', 'Valor': resultado['registros_con_diferencias']},
            {'Métrica': 'Columnas solo en PROD',     'Valor': ', '.join(resultado['cols_solo_prod']) or 'Ninguna'},
            {'Métrica': 'Columnas solo en DEV',      'Valor': ', '.join(resultado['cols_solo_dev'])  or 'Ninguna'},
            {'Métrica': 'Tipos distintos',           'Valor': resultado['tipos_distintos']},
            {'Métrica': 'Error',                     'Valor': resultado['error'] or ''},
        ])
        print(f"  [{_ts()}] Escribiendo resumen en Sheets...")
        update_sheets_in_drive_folder_chunked(gc, ID_HOJA_RESULTADOS, hoja_resumen, df_resumen)
        print(f"  [{_ts()}] ✅ Escrito resumen en '{hoja_resumen}'")

        elapsed = int(time.time() - t0)
        print(f"  Tiempo total proyecto: {elapsed}s")

    except Exception as e:
        resultado['error'] = traceback.format_exc()
        print(f"  [{_ts()}] ❌ ERROR: {e}")

    resumen_global.append(resultado)

print(f"\n{'='*70}")
print(f"  LOOP FINALIZADO  —  {_ts()}")

# Resumen Final

In [ ]:
W = 42  # ancho columna proyecto

print("\n" + "=" * 110)
print("  RESUMEN GLOBAL DE VALIDACIONES")
print("=" * 110)
print(f"  {'PROYECTO':<{W}} {'IGUALES':<8} {'F.PROD':<8} {'F.DEV':<8} "
      f"{'DUP.P':<7} {'DUP.D':<7} {'S.PROD':<8} {'S.DEV':<8} {'DIFS':<6} {'ESTADO'}")
print(f"  {'-'*107}")

for r in resumen_global:
    if r['error']:
        estado = "❌ ERROR"
    elif r['identicos']:
        estado = "✅ OK"
    elif r['registros_con_diferencias'] == 0 and r['solo_en_prod'] == 0 and r['solo_en_dev'] == 0:
        estado = "⚠ SIN DIFS"
    else:
        estado = "⚠ CON DIFS"

    print(f"  {r['proyecto']:<{W}} {'SI' if r['identicos'] else 'NO':<8} "
          f"{r['filas_prod']:<8} {r['filas_dev']:<8} "
          f"{r['dups_prod']:<7} {r['dups_dev']:<7} "
          f"{r['solo_en_prod']:<8} {r['solo_en_dev']:<8} "
          f"{r['registros_con_diferencias']:<6} {estado}")

    if r['cols_solo_prod']:
        print(f"    {'':>{W}} Cols solo PROD: {r['cols_solo_prod']}")
    if r['cols_solo_dev']:
        print(f"    {'':>{W}} Cols solo DEV:  {r['cols_solo_dev']}")
    if r['error']:
        lineas_error = r['error'].strip().split('\n')
        print(f"    {'':>{W}} {lineas_error[-1]}")

print("=" * 110)

ok    = sum(1 for r in resumen_global if r['identicos'] and not r['error'])
difs  = sum(1 for r in resumen_global if not r['identicos'] and not r['error'])
errs  = sum(1 for r in resumen_global if r['error'])

print(f"\n  Total: {len(resumen_global)}  |  ✅ Idénticos: {ok}  |  ⚠ Con diferencias: {difs}  |  ❌ Errores: {errs}")